# Markdown Cleaner Test

This notebook tests the `clean_markdown_for_rag()` function from the markdown cleaner.

In [ ]:
import sys
sys.path.insert(0, '..')

from markdown_cleaner import clean_markdown_for_rag
from vector_engine import chunk_text

print('Imports OK')

## Test 1: Basic Markdown Cleanup

In [ ]:
sample = """# Project

![badge](https://img.shields.io/badge/Python-3776AB)
<img src="logo.png" width="200"/>

This is **important** content about the project. It uses emojis 🚀 and 😊.

<p align="center"><img src="demo.gif"/></p>

## Features

- Feature 1: does something
- Feature 2: does another thing
"""

print('=== BEFORE ===')
print(repr(sample[:200]))
print()
cleaned = clean_markdown_for_rag(sample)
print('=== AFTER ===')
print(cleaned)

## Test 2: Empty / Edge Cases

In [ ]:
# Empty string
assert clean_markdown_for_rag('') == ''
print('Empty: OK')

# No markdown, plain text
assert clean_markdown_for_rag('Hello world') == 'Hello world'
print('Plain text: OK')

# Only images
result = clean_markdown_for_rag('![img](url.png)')
print(f'Only images: {repr(result)}')

# Emoji only
result = clean_markdown_for_rag('🚀😊')
print(f'Emoji only: {repr(result)}')
print('Edge cases done')

## Test 3: Real GitHub Profile README

In [ ]:
import httpx

# Fetch the profile README from GitHub
url = 'https://api.github.com/repos/ridhwanrazaliwork/ridhwanrazaliwork/readme'
resp = httpx.get(url, headers={
    'Accept': 'application/vnd.github.raw+json',
    'Authorization': 'Bearer ' + ('sk-or-v1-fake')  # unauthed works for public repos
})
if resp.status_code == 200:
    raw = resp.text
    print(f'Raw length: {len(raw)} chars')
    cleaned = clean_markdown_for_rag(raw)
    print(f'Cleaned length: {len(cleaned)} chars')
    print(f'Reduction: {len(raw) - len(cleaned)} chars')
    print()
    print('=== CLEANED PREVIEW ===')
    print(cleaned[:1000])
else:
    print('Could not fetch profile README:', resp.status_code)
    print('Using local sample instead...')

## Test 4: Chunking with Metadata Injection

In [ ]:
sample = """# Project

## Overview
This is a demo project.

## Installation
Run `pip install` to get started.

## Usage
Execute the script.
"""

chunks = chunk_text(sample, 'demo-repo', updated_at='2026-06-15T10:30:00Z')

print(f'Total chunks: {len(chunks)}')
print()
for chunk_id, text, meta in chunks:
    print(f'--- {chunk_id} ---')
    print(text)
    print()

## Test 5: Chunking Without updated_at (Fallback)

In [ ]:
chunks_no_meta = chunk_text(sample, 'demo-repo')
for chunk_id, text, meta in chunks_no_meta[:1]:
    print(f'--- {chunk_id} ---')
    print(text)
    print()
    print('No metadata suffix (None updated_at)')